In [ ]:
!pip install anthropic
!pip install google-generativeai
!pip install transformers

In [ ]:
# Python standard library
import os

# Third-party libraries
import anthropic
from dotenv import load_dotenv
import google.genai
import pandas as pd
import tiktoken
from transformers import AutoTokenizer

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
DIR = "/content/drive/MyDrive/Colab Notebooks/MNA/Proyecto Integrador/clear-equity-cards"
os.chdir(DIR)

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

# Sobrescribimos la variable de entorno para Colab DESPUÉS de cargar el .env
os.environ["PROJECT_ROOT"] = "/content/drive/MyDrive/Colab Notebooks/MNA/Proyecto Integrador/clear-equity-cards"

PROJECT_ROOT = os.getenv("PROJECT_ROOT")
if not PROJECT_ROOT:
    raise EnvironmentError("PROJECT_ROOT is not set. Check your .env file.")

os.chdir(PROJECT_ROOT)
print(f"Directorio de trabajo actual: {os.getcwd()}")


Directorio de trabajo actual: /content/drive/MyDrive/Colab Notebooks/MNA/Proyecto Integrador/clear-equity-cards


In [ ]:
hf_models = pd.read_table("01_data_processed/hf_models_by_tokenizer.tsv")
# hf_model_list = hf_models["tokenizer_repo_id"].tolist()

# Using 3 tokenizers as requested
hf_model_list = ["Qwen/Qwen3.6-35B-A3B", "Qwen/Qwen3-8B", "deepseek-ai/DeepSeek-R1-0528-Qwen3-8B"]

hf_remote_code_list = []

openai_model_list = []

anthropic_model_list = []

google_model_list = []


In [ ]:
# Let's find all Qwen-related models in your dataset to get the correct Hugging Face IDs
qwen_models = hf_models[hf_models['tokenizer_repo_id'].astype(str).str.contains('Qwen', case=False, na=False)]
display(qwen_models['tokenizer_repo_id'].tolist())

['deepseek-ai/DeepSeek-R1-0528-Qwen3-8B',
 'Qwen/Qwen3.6-35B-A3B',
 'deepseek-ai/DeepSeek-R1-Distill-Qwen-7B',
 'Qwen/Qwen3-8B']

In [ ]:
# Load only the first 10 languages for the analysis
flores200_dev = pd.read_table("01_data_processed/flores200_dev.tsv").head(10)


In [ ]:
if os.path.exists("02_output_model_experiments/flores200_token_counts.csv"):
    old_results_df = pd.read_csv("02_output_model_experiments/flores200_token_counts.csv")
    model_list = old_results_df["model"].unique().tolist()
        # get models for which token counts have already been collected

    models_to_drop = list(set(model_list) - set(hf_model_list + openai_model_list + anthropic_model_list + google_model_list))
        # find eliminated models
    if len(models_to_drop) > 0:
        old_results_df = old_results_df.loc[old_results_df["model"].isin(models_to_drop) == False,]
            # drop eliminated models from data

    hf_model_list = list(set(hf_model_list) - set(model_list))
    openai_model_list = list(set(openai_model_list) - set(model_list))
    anthropic_model_list = list(set(anthropic_model_list) - set(model_list))
    google_model_list = list(set(google_model_list) - set(model_list))
        # keep newly added models
else:
    old_results_df = pd.DataFrame()

In [ ]:
results = []

# Hugging Face open-source models
for model in hf_model_list:
    print(f"Working on model: {model}")
    if model in hf_remote_code_list:
        tokenizer = AutoTokenizer.from_pretrained(model, trust_remote_code = True)
    else:
        tokenizer = AutoTokenizer.from_pretrained(model)

    for row_index, row_data in flores200_dev.iterrows():
        file = row_data["file"]
        text = row_data["text"]

        # Tokenize data
        tokens = tokenizer.encode(text)
        token_count = len(tokens)

        # Append results for this iteration in results list
        result_i = {"model": model, "file": file, "token_count": token_count}
        results.append(result_i)

# OpenAI closed-source models
for model in openai_model_list:
    print(f"Working on model: {model}")
    tokenizer = tiktoken.encoding_for_model(model)

    for row_index, row_data in flores200_dev.iterrows():
        file = row_data["file"]
        text = row_data["text"]

        # Tokenize data
        tokens = tokenizer.encode(text)
        token_count = len(tokens)

        # Append results for this iteration in results list
        result_i = {"model": model, "file": file, "token_count": token_count}
        results.append(result_i)

# Anthropic closed-source models
client = anthropic.Anthropic()
for model in anthropic_model_list:
    print(f"Working on model: {model}")

    for row_index, row_data in flores200_dev.iterrows():
        file = row_data["file"]
        text = row_data["text"]

        # Tokenize data
        response = client.messages.count_tokens(model = model, messages = [{"role": "user", "content": text}])
        token_count = response.input_tokens

        # Append results for this iteration in results list
        result_i = {"model": model, "file": file, "token_count": token_count}
        results.append(result_i)

# Google closed-source models
client = google.genai.Client()
for model in google_model_list:
    print(f"Working on model: {model}")

    for row_index, row_data in flores200_dev.iterrows():
        file = row_data["file"]
        text = row_data["text"]

        # Tokenize data
        response = client.models.count_tokens(model = model, contents = text)
        token_count = response.total_tokens

        # Append results for this iteration in results list
        result_i = {"model": model, "file": file, "token_count": token_count}
        results.append(result_i)

# Create results DataFrame
new_results_df = pd.DataFrame(data = results)
flores200_token_counts = pd.concat(objs = [old_results_df, new_results_df], ignore_index = True)

In [ ]:
flores200_token_counts.to_csv("02_output_model_experiments/qwen_flores200_token_counts.csv", index = False)

In [ ]:
flores200_langinfo = pd.read_table("01_data_processed/flores200_langinfo.tsv", keep_default_na = False, na_values = [""])

flores200_token_counts_langinfo = pd.merge(left = flores200_token_counts, right = flores200_langinfo, how = "outer", on = "file", indicator = True)

assert all(flores200_token_counts_langinfo["_merge"] == "both")
    # confirming merge resulted in full match
flores200_token_counts_langinfo = flores200_token_counts_langinfo.drop(columns = "_merge")

assert flores200_token_counts_langinfo["speakers"].isna().sum() == 0
    # no more missing values in "speakers" column

In [ ]:
assert all(flores200_token_counts_langinfo.dtypes.astype(str).isin(["object", "string", "int64", "float64"]))
    # confirming all columns are either of type object, string, int64, or float64
for c in flores200_token_counts_langinfo.columns:
    if flores200_token_counts_langinfo[c].dtype in ["object", "string"]:
        assert flores200_token_counts_langinfo[c].astype(str).str.contains(r",").sum() == 0
    # confirming there are no commas in string columns - safe to save as CSV

flores200_token_counts_langinfo.to_csv("02_output_model_experiments/qwen_flores200_token_counts_langinfo.csv", index = False)


# TODO: try roundtrip tokenization to make sure the text is tokenized correctly (for deepseek llama issue)
